Tomado de:
A Quick Introduction to PyTorch: Using Deep Learning for Stock Price Prediction
Published: February 23, 2022
5 min read
Written by: H2O.ai Team

https://h2o.ai/blog/2022/a-quick-introduction-to-pytorch-using-deep-learning-for-stock-price-prediction/$0

In [0]:
!pip install statsmodels
!pip install linearmodels
!pip install lseg.data
!pip install pandas_datareader
!pip install torch

In [0]:
import lseg.data as ld
import numpy as np
import pandas as pd

from datetime import datetime, timedelta

In [0]:
# Fetch Apple stock prices for the last year using ld API
# Retrieve credentials securely from Databricks Secrets
APP_KEY = dbutils.secrets.get(scope="refinitiv_scope", key="app_key")
USERNAME = dbutils.secrets.get(scope="refinitiv_scope", key="username")
PASSWORD = dbutils.secrets.get(scope="refinitiv_scope", key="password")

# Open platform session with credentials from secrets
session = ld.session.platform.Definition(
    app_key=APP_KEY,
    grant=ld.session.platform.GrantPassword(
        username=USERNAME,
        password=PASSWORD
    ),
    signon_control=True  # Allow multiple concurrent sessions
).get_session()

# Open the session
result = session.open()
print(f"Session opened successfully: {result}")

# Set as default session
ld.session.set_default(session)
print("Session set as default")

In [0]:
# Calculate date range for last year
end_date = datetime.now().strftime("%Y-%m-%d")
start_date = (datetime.now() - timedelta(days=365)).strftime("%Y-%m-%d")

# Fetch historical data using ld.get_history (not get_data)
df = ld.get_history(

    universe="AAPL.O",
    fields=["TR.PriceClose.date", "TR.PriceClose", "TR.PriceOpen", "TR.PriceHigh", "TR.PriceLow", "TR.Volume"],
    interval="1D",
    start=start_date,
    end=end_date
)

# Display the result
display(df.head())

In [0]:
# Rename columns to standard names
df = df.rename(columns={
    'Price Close': 'close',
    'Price Open': 'open',
    'Price High': 'high',
    'Price Low': 'low'
})

display(df.head())

In [0]:
import matplotlib.pyplot as plt

plt.plot(df.open.values, color='red', label='open')
plt.plot(df.close.values, color='green', label='close')
plt.plot(df.low.values, color='blue', label='low')
plt.plot(df.high.values, color='black', label='high')
plt.title('stock price')
plt.xlabel('time [days]')
plt.ylabel('price')
plt.legend(loc='best')

In [0]:
import sklearn.preprocessing

min_max_scaler = sklearn.preprocessing.MinMaxScaler()

df['open'] = min_max_scaler.fit_transform(df.open.values.reshape(-1,1))
df['high'] = min_max_scaler.fit_transform(df.high.values.reshape(-1,1))
df['low'] = min_max_scaler.fit_transform(df.low.values.reshape(-1,1))
df['close'] = min_max_scaler.fit_transform(df['close'].values.reshape(-1,1))
data = df[['open','close','low','high']].values

In [0]:
data

In [0]:
seq_len=20
sequences=[]
for index in range(len(data) - seq_len): 
 sequences.append(data[index: index + seq_len])
sequences= np.array(sequences)

In [0]:
sequences

In [0]:
valid_set_size_percentage = 10 
test_set_size_percentage = 10 

valid_set_size = int(np.round(valid_set_size_percentage/100*sequences.shape[0])) 
test_set_size = int(np.round(test_set_size_percentage/100*sequences.shape[0]))
train_set_size = sequences.shape[0] - (valid_set_size + test_set_size)

x_train = sequences[:train_set_size,:-1,:]
y_train = sequences[:train_set_size,-1,:]

x_valid = sequences[train_set_size:train_set_size+valid_set_size,:-1,:]
y_valid = sequences[train_set_size:train_set_size+valid_set_size,-1,:]

x_test = sequences[train_set_size+valid_set_size:,:-1,:]
y_test = sequences[train_set_size+valid_set_size:,-1,:]

In [0]:
import torch
from torch.utils.data import TensorDataset, DataLoader

x_train = torch.tensor(x_train).float()
y_train = torch.tensor(y_train).float()

x_valid = torch.tensor(x_valid).float()
y_valid = torch.tensor(y_valid).float()

train_dataset = TensorDataset(x_train,y_train)
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)

valid_dataset = TensorDataset(x_valid,y_valid)
valid_dataloader = DataLoader(valid_dataset, batch_size=32, shuffle=True)

In [0]:
from torch import nn

class NeuralNetwork(nn.Module):
  def __init__(self):
    super(NeuralNetwork, self).__init__()
    self.lstm = nn.LSTM(4,64,batch_first=True)
    self.fc = nn.Linear(64,4)

  def forward(self, x):
    output, (hidden, cell) = self.lstm(x)
    x = self.fc(hidden)
    return x
 
model = NeuralNetwork()

#push to cuda if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') 
model = model.to(device)

In [0]:
import torch.optim as optim
optimizer = optim.Adam(model.parameters())
mse = nn.MSELoss()

In [0]:
def train(dataloader):
  epoch_loss = 0
  model.train() 

  for batch in dataloader:
    optimizer.zero_grad() 
    x,y= batch
    pred = model(x)
    loss = mse(pred[0],y) 
    loss.backward() 
    optimizer.step() 
    epoch_loss += loss.item() 
  return epoch_loss

In [0]:
def evaluate(dataloader):
  epoch_loss = 0
  model.eval() 

  with torch.no_grad():
    for batch in dataloader: 
      x,y= batch
      pred = model(x)
      loss = mse(pred[0],y) 
      epoch_loss += loss.item() 
  return epoch_loss / len(dataloader)

In [0]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
x_train = x_train.to(device)
y_train = y_train.to(device)
x_valid = x_valid.to(device)
y_valid = y_valid.to(device)

train_dataset = TensorDataset(x_train, y_train)
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)

valid_dataset = TensorDataset(x_valid, y_valid)
valid_dataloader = DataLoader(valid_dataset, batch_size=32, shuffle=True)

n_epochs = 50
best_valid_loss = float('inf')

for epoch in range(n_epochs):
  train_loss = train(train_dataloader)
  valid_loss = evaluate(valid_dataloader)

  #save the best model
  if valid_loss < best_valid_loss:
    best_valid_loss = valid_loss
    torch.save(model, 'saved_weights.pt')
    print("Epoch ",epoch+1)
  print(f'\tTrain Loss: {train_loss:.5f}')
print(f'\tVal Loss: {valid_loss:.5f}\n')

In [0]:
model=torch.load('saved_weights.pt', weights_only=False)

In [0]:
x_test= torch.tensor(x_test).float()
x_test = x_test.to(next(model.parameters()).device)

with torch.no_grad():
  y_test_pred = model(x_test)

y_test_pred = y_test_pred.cpu().numpy()[0]

In [0]:
idx=0

plt.plot(np.arange(y_train.shape[0], y_train.shape[0]+y_test.shape[0]),
 y_test[:,idx], color='black', label='test target')

plt.plot(np.arange(y_train.shape[0], y_train.shape[0]+y_test_pred.shape[0]),
 y_test_pred[:,idx], color='green', label='test prediction')

plt.title('future stock prices')
plt.xlabel('time [days]')
plt.ylabel('normalized price')
plt.legend(loc='best')

In [0]:
def calculate_metrics(y_true, y_pred, column_names=None):
    """
    Calcula MAE, MSE, RMSE, MAPE y sMAPE entre los valores reales y las predicciones,
    evaluando cada columna de forma independiente.
    
    Parameters
    ----------
    y_true : np.ndarray
        Valores reales. Forma (n_samples,) para una sola columna o (n_samples, n_cols) para varias.
    y_pred : np.ndarray
        Valores predichos. Misma forma que y_true.
    column_names : list[str] or None
        Nombres de las columnas. Si es None se usan índices numéricos ('col_0', 'col_1', ...).
    
    Returns
    -------
    dict
        Diccionario anidado: {columna: {'MAE': ..., 'MSE': ..., 'RMSE': ..., 'MAPE': ..., 'sMAPE': ...}}.
        Si solo hay una columna, el diccionario tiene una sola entrada bajo la clave correspondiente.
    """
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)

    # Asegurar formato 2D (n_samples, n_cols)
    if y_true.ndim == 1:
        y_true = y_true.reshape(-1, 1)
    if y_pred.ndim == 1:
        y_pred = y_pred.reshape(-1, 1)

    n_cols = y_true.shape[1]

    if column_names is None:
        column_names = [f'col_{i}' for i in range(n_cols)]
    elif len(column_names) != n_cols:
        raise ValueError(f"column_names tiene {len(column_names)} elementos pero y_true tiene {n_cols} columnas.")

    results = {}

    for j, col_name in enumerate(column_names):
        yt = y_true[:, j]
        yp = y_pred[:, j]

        # MAE - Mean Absolute Error
        mae = np.mean(np.abs(yt - yp))

        # MSE - Mean Squared Error
        mse = np.mean((yt - yp) ** 2)

        # RMSE - Root Mean Squared Error
        rmse = np.sqrt(mse)

        # MAPE - Mean Absolute Percentage Error
        # Evita división por cero usando solo donde y_true != 0
        mask_mape = yt != 0
        if np.any(mask_mape):
            mape = np.mean(np.abs((yt[mask_mape] - yp[mask_mape]) / yt[mask_mape])) * 100
        else:
            mape = np.nan

        # sMAPE - Symmetric Mean Absolute Percentage Error
        denominator = np.abs(yt) + np.abs(yp)
        mask_smape = denominator != 0
        if np.any(mask_smape):
            smape = np.mean(
                np.abs(yt[mask_smape] - yp[mask_smape]) / denominator[mask_smape]
            ) * 100
        else:
            smape = np.nan

        results[col_name] = {
            'MAE': mae,
            'MSE': mse,
            'RMSE': rmse,
            'MAPE': mape,
            'sMAPE': smape
        }

    return results

# Ejemplo de uso con los datos de prueba (varias columnas)
#column_names = ['open', 'close', 'low', 'high']
#metrics = calculate_metrics(y_test, y_test_pred, column_names=column_names)
#for col_name, col_metrics in metrics.items():
#    print(f'--- {col_name} ---')
#    for metric_name, value in col_metrics.items():
#        print(f'  {metric_name}: {value:.6f}')

In [0]:
y_test

In [0]:
y_test_pred

In [0]:
# Remove rows with NaN values before calculating metrics
mask = ~np.isnan(y_test).any(axis=1)
y_test_clean = y_test[mask]
y_test_pred_clean = y_test_pred[mask]

metrics = calculate_metrics(y_test_clean, y_test_pred_clean)
for col_name, col_metrics in metrics.items():
    print(f'--- {col_name} ---')
    for metric_name, value in col_metrics.items():
        print(f'  {metric_name}: {value:.6f}')

In [0]:
# Modelo Benchmark
Modelo de redes neuronales LSTM
Modelos tradicionales AR, MA, ARMA, ARIMA, ARIMAX, SARIMA, SARIMAX, Holt-Winters, Holt-Winters con tendencia y estacionalidad, Holt-Winters con tendencia, Holt-Winters con estacionalidad, Holt-Winters con tendencia y estacional
Modelo ingenuo Tomo estimado el último valor disponible 

In [0]:
# Modelo ingenuo (naive): la predicción de hoy es el valor real del día anterior
y_test_naive = y_test[:-1]        # valores reales en t-1
y_test_actual = y_test[1:]         # valores reales en t (objetivo real)

# Remove rows with NaN values before calculating metrics
mask_naive = ~np.isnan(y_test_naive).any(axis=1) & ~np.isnan(y_test_actual).any(axis=1)
y_test_naive_clean = y_test_naive[mask_naive]
y_test_actual_clean = y_test_actual[mask_naive]

metrics_naive = calculate_metrics(y_test_actual_clean, y_test_naive_clean,
                                  column_names=['open', 'close', 'low', 'high'])

print('=== Modelo Ingenuo (naive: valor del día anterior) ===')
for col_name, col_metrics in metrics_naive.items():
    print(f'--- {col_name} ---')
    for metric_name, value in col_metrics.items():
        print(f'  {metric_name}: {value:.6f}')

In [0]:
import pandas as pd

# --- LSTM metrics (ya calculadas en la celda anterior) ---
metrics_lstm = calculate_metrics(y_test_clean, y_test_pred_clean,
                                 column_names=['open', 'close', 'low', 'high'])

# --- Naive metrics (ya calculadas en la celda anterior) ---
# metrics_naive ya está disponible desde la celda del modelo ingenuo

# Construir filas de la tabla comparativa
rows = []
for col in ['open', 'close', 'low', 'high']:
    for metric in ['MAE', 'MSE', 'RMSE', 'MAPE', 'sMAPE']:
        rows.append({
            'Columna': col,
            'Métrica': metric,
            'LSTM': metrics_lstm[col][metric],
            'Ingenuo (naive)': metrics_naive[col][metric],
            'Diferencia (LSTM - Naive)': metrics_lstm[col][metric] - metrics_naive[col][metric]
        })

df_comparison = pd.DataFrame(rows)

# Formatear la tabla para mejor legibilidad
pd.set_option('display.float_format', '{:.6f}'.format)
print('=== Comparación de modelos: LSTM vs Ingenuo (naive) ===\n')
display(df_comparison)

In [0]:
import mlflow
import mlflow.pytorch
from mlflow.models import infer_signature
import matplotlib.pyplot as plt
import io
from PIL import Image

# Set experiment name
mlflow.set_experiment("/Users/nvaldez@tec.mx/finanzas_2026_agosto/LSTM_Stock_Prediction")

# Start MLflow run
with mlflow.start_run(run_name="LSTM_Stock_Price_Forecasting") as run:
    
    # Log parameters
    mlflow.log_param("seq_len", seq_len)
    mlflow.log_param("batch_size", 32)
    mlflow.log_param("n_epochs", 50)
    mlflow.log_param("lstm_hidden_size", 64)
    mlflow.log_param("optimizer", "Adam")
    mlflow.log_param("loss_function", "MSE")
    mlflow.log_param("train_set_size", train_set_size)
    mlflow.log_param("valid_set_size", valid_set_size)
    mlflow.log_param("test_set_size", test_set_size)
    mlflow.log_param("device", str(device))
    
    # Recreate model and optimizer for clean training
    model = NeuralNetwork()
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    optimizer = optim.Adam(model.parameters())
    mse = nn.MSELoss()
    
    # Move data to device
    x_train = x_train.to(device)
    y_train = y_train.to(device)
    x_valid = x_valid.to(device)
    y_valid = y_valid.to(device)
    
    train_dataset = TensorDataset(x_train, y_train)
    train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    
    valid_dataset = TensorDataset(x_valid, y_valid)
    valid_dataloader = DataLoader(valid_dataset, batch_size=32, shuffle=True)
    
    # Training loop with metric logging
    n_epochs = 50
    best_valid_loss = float('inf')
    train_losses = []
    valid_losses = []
    
    for epoch in range(n_epochs):
        train_loss = train(train_dataloader)
        valid_loss = evaluate(valid_dataloader)
        
        train_losses.append(train_loss)
        valid_losses.append(valid_loss)
        
        # Log metrics to MLflow
        mlflow.log_metric("train_loss", train_loss, step=epoch)
        mlflow.log_metric("valid_loss", valid_loss, step=epoch)
        
        # Save best model
        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            torch.save(model, 'saved_weights.pt')
            print(f"Epoch {epoch+1} - New best model saved")
            print(f'\tTrain Loss: {train_loss:.5f}')
            print(f'\tVal Loss: {valid_loss:.5f}\n')
    
    # Log best validation loss
    mlflow.log_metric("best_valid_loss", best_valid_loss)
    
    # === Generate and log training curve ===
    fig_training = plt.figure(figsize=(10, 6))
    plt.plot(train_losses, label='Train Loss', color='blue')
    plt.plot(valid_losses, label='Validation Loss', color='orange')
    plt.xlabel('Epoch')
    plt.ylabel('Loss (MSE)')
    plt.title('Training and Validation Loss Over Epochs')
    plt.legend()
    plt.grid(True)
    mlflow.log_figure(fig_training, "training_validation_loss.png")
    plt.close()
    
    # === Load best model and generate predictions ===
    model = torch.load('saved_weights.pt', weights_only=False)
    
    x_test_tensor = torch.tensor(x_test).float()
    x_test_tensor = x_test_tensor.to(next(model.parameters()).device)
    
    with torch.no_grad():
        y_test_pred_tensor = model(x_test_tensor)
    
    y_test_pred = y_test_pred_tensor.cpu().numpy()[0]
    
    # Remove NaN rows
    mask = ~np.isnan(y_test).any(axis=1)
    y_test_clean = y_test[mask]
    y_test_pred_clean = y_test_pred[mask]
    
    # === Calculate metrics (LSTM) ===
    metrics_lstm = calculate_metrics(y_test_clean, y_test_pred_clean,
                                     column_names=['open', 'close', 'low', 'high'])
    
    # Log LSTM test metrics
    for col_name, col_metrics in metrics_lstm.items():
        for metric_name, value in col_metrics.items():
            mlflow.log_metric(f"test_{col_name}_{metric_name}", value)
    
    # === Generate and log prediction plot ===
    fig_pred = plt.figure(figsize=(10, 6))
    idx = 0  # Plot first column (open)
    plt.plot(np.arange(y_train.shape[0], y_train.shape[0]+y_test_clean.shape[0]),
             y_test_clean[:, idx], color='black', label='test target')
    plt.plot(np.arange(y_train.shape[0], y_train.shape[0]+y_test_pred_clean.shape[0]),
             y_test_pred_clean[:, idx], color='green', label='test prediction')
    plt.title('Future Stock Prices (LSTM Predictions)')
    plt.xlabel('Time [days]')
    plt.ylabel('Normalized Price')
    plt.legend(loc='best')
    mlflow.log_figure(fig_pred, "lstm_predictions.png")
    plt.close()
    
    # === Calculate naive baseline metrics ===
    y_test_naive = y_test[:-1]
    y_test_actual = y_test[1:]
    mask_naive = ~np.isnan(y_test_naive).any(axis=1) & ~np.isnan(y_test_actual).any(axis=1)
    y_test_naive_clean = y_test_naive[mask_naive]
    y_test_actual_clean = y_test_actual[mask_naive]
    
    metrics_naive = calculate_metrics(y_test_actual_clean, y_test_naive_clean,
                                      column_names=['open', 'close', 'low', 'high'])
    
    # Log naive baseline metrics
    for col_name, col_metrics in metrics_naive.items():
        for metric_name, value in col_metrics.items():
            mlflow.log_metric(f"naive_{col_name}_{metric_name}", value)
    
    # === Create comparison table ===
    import pandas as pd
    rows = []
    for col in ['open', 'close', 'low', 'high']:
        for metric in ['MAE', 'MSE', 'RMSE', 'MAPE', 'sMAPE']:
            rows.append({
                'Columna': col,
                'Métrica': metric,
                'LSTM': metrics_lstm[col][metric],
                'Ingenuo (naive)': metrics_naive[col][metric],
                'Diferencia (LSTM - Naive)': metrics_lstm[col][metric] - metrics_naive[col][metric]
            })
    
    df_comparison = pd.DataFrame(rows)
    
    # Save and log comparison table
    comparison_path = "model_comparison.csv"
    df_comparison.to_csv(comparison_path, index=False)
    mlflow.log_artifact(comparison_path)
    
    # Display comparison table
    pd.set_option('display.float_format', '{:.6f}'.format)
    print('\n=== Comparación de modelos: LSTM vs Ingenuo (naive) ===\n')
    display(df_comparison)
    
    # === Log the model with signature ===
    # Prepare signature
    signature_input = x_test_tensor.cpu()[:3]  # First 3 test samples
    signature_output = model(signature_input.to(device)).cpu().detach().numpy()[0][:3]
    signature = infer_signature(signature_input.numpy(), signature_output)
    
    # Log model
    model_info = mlflow.pytorch.log_model(
        model,
        artifact_path="model",
        signature=signature,
        input_example=signature_input.numpy()
    )
    
    print(f"\n✓ MLflow run completed: {run.info.run_id}")
    print(f"✓ Model logged at: {model_info.model_uri}")
    print(f"\n📊 Experiment: /Users/nvaldez@tec.mx/finanzas_2026_agosto/LSTM_Stock_Prediction")